# Show-o2 Understanding and Generation

Follow the generated prompt into the image stage. Which input details survive?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://github.com/showlab/Show-o/tree/main/show-o2

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "transformers==4.51.3" "huggingface-hub==0.36.2" "accelerate" "safetensors" "diffusers==0.30.3" "omegaconf" "wandb==0.31.0" "einops" "sentencepiece" "protobuf" "timm==1.0.22" "torchdiffeq"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import os, random, tarfile, requests, time, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"🖥️ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name()}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   CPU execution is slower; the same model experiments will run.")

IMG_DIR = "./generation-pets"
CACHE_DIR = "./cache"
OUT_DIR = "./lab5_outputs"
for d in [IMG_DIR, CACHE_DIR, OUT_DIR, f"{IMG_DIR}/dog", f"{IMG_DIR}/cat"]:
    os.makedirs(d, exist_ok=True)

## Input images

Fixed Oxford-IIIT Pet classroom subset. Source: https://www.robots.ox.ac.uk/~vgg/data/pets/

In [ ]:
from pathlib import Path
import urllib.request, zipfile

IMG_DIR = "./data"
NUM_IMAGES_PER_CLASS = 50
batch_size = 16
if not all(len(list((Path(IMG_DIR) / label).glob("*.jpg"))) == 50 for label in ["dog", "cat"]):
    url = "https://moai.postech.ac.kr/teaching/27s-programming-for-ax/labs/colab/oxford-pets-classroom.zip"
    archive, _ = urllib.request.urlretrieve(url)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(".")
image_paths, image_labels = [], []
for label in ["dog", "cat"]:
    paths = sorted((Path(IMG_DIR) / label).glob("*.jpg"))
    image_paths.extend(str(path) for path in paths)
    image_labels.extend([label] * len(paths))
print(f"Classroom subset: {len(image_paths)} images")


In [ ]:
def show_image_grid(images, titles=None, cols=4, figsize=(14, 7), suptitle=None):
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for i, ax in enumerate(axes):
        ax.axis("off")
        if i < len(images):
            img = images[i]
            if isinstance(img, (str, Path)):
                img = Image.open(img).convert("RGB")
            ax.imshow(img)
            if titles is not None:
                ax.set_title(titles[i], fontsize=11, fontweight="bold")
    if suptitle:
        fig.suptitle(suptitle, fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

n_show = min(8, len(image_paths))
show_image_grid(
    image_paths[:n_show],
    titles=[y.upper() for y in image_labels[:n_show]],
    cols=4,
    figsize=(14, 7),
    suptitle="🐶🐱 Sample Images for Lab 5",
)

In [ ]:
# Select one real pet image as the shared multimodal input.
PET_IMAGE_INDEX = 0

pet_image_path = image_paths[PET_IMAGE_INDEX]
pet_label = image_labels[PET_IMAGE_INDEX]
pet_image = Image.open(pet_image_path).convert('RGB')

plt.figure(figsize=(5, 5))
plt.imshow(pet_image)
plt.axis('off')
plt.title(f"Reference pet image: {pet_label}")
plt.show()

print("Reference image path:", pet_image_path)
print("Ground-truth label:", pet_label)


In [ ]:
import os, subprocess, sys, json, shutil
from pathlib import Path

RUN_SHOWO2 = True
SHOWO_REVISION = "45a5a2de01d1ebd10cd5864d29310a76476cdf23"
SHOWO_ROOT = Path.home() / ".cache" / "ax-model-labs" / "Show-o"
SHOWO2_DIR = SHOWO_ROOT / "show-o2"
showo_env = Path.home() / ".venvs" / "ax-showo2" / "bin" / "python"
SHOWO_PYTHON = sys.executable
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Show-o2 runtime:", SHOWO_PYTHON)
print("Show-o2 device:", "cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
if not SHOWO_ROOT.exists():
    SHOWO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/showlab/Show-o.git", str(SHOWO_ROOT)], check=True)
subprocess.run(["git", "-C", str(SHOWO_ROOT), "checkout", SHOWO_REVISION], check=True)
actual_revision = subprocess.check_output(["git", "-C", str(SHOWO_ROOT), "rev-parse", "HEAD"], text=True).strip()
assert actual_revision == SHOWO_REVISION
print("Pinned official Show-o2 source:", actual_revision)


In [ ]:
# This Colab runtime was prepared in the setup cell.
# Do not install or upgrade packages in a running kernel.
subprocess.run([SHOWO_PYTHON, "-c", "import torch, transformers, diffusers, timm, wandb; print(torch.__version__, transformers.__version__, diffusers.__version__)"], check=True)
from huggingface_hub import hf_hub_download
vae_checkpoint = hf_hub_download("Wan-AI/Wan2.1-T2V-14B", "Wan2.1_VAE.pth")
SHOWO_VAE = Path(vae_checkpoint).resolve()
print("Wan2.1 VAE is ready.")


In [ ]:
SHOWO2_OUTPUT_DIR = Path("./showo2_outputs")
SHOWO2_OUTPUT_DIR.mkdir(exist_ok=True)

# The official T2I script reads prompts from a text file.
PROMPT_FILE = Path("./showo2_prompts.txt")

print("Output directory:", SHOWO2_OUTPUT_DIR.resolve())
print("Prompt file:", PROMPT_FILE.resolve())


In [ ]:
#!/usr/bin/env python3
"""Adapt the pinned official image inference scripts to the classroom CPU runtime.

Attention math and trained weights stay upstream. CPU uses the official dense
attention masks; FlexAttention entry points fail explicitly if accidentally used.
"""
import argparse
from pathlib import Path
import subprocess

REVISION = '45a5a2de01d1ebd10cd5864d29310a76476cdf23'


def prepare(root):
    revision = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
    if revision != REVISION:
        raise RuntimeError(f'Expected Show-o revision {REVISION}, found {revision}')
    folder = root / 'show-o2'
    compatibility = '''try:
    from torch.nn.attention.flex_attention import BlockMask, flex_attention, create_block_mask
except ImportError:
    class BlockMask:
        pass
    def flex_attention(*args, **kwargs):
        raise RuntimeError("FlexAttention requires a newer PyTorch. Use the dense inference path on CPU.")
    create_block_mask = flex_attention
'''
    (folder / 'ax_attention_compat.py').write_text(compatibility)
    for name in ['models/qwen2.py', 'models/modules.py', 'models/omni_attention.py',
                 'models/modeling_showo2_qwen2_5.py', 'inference_mmu.py', 'inference_t2i.py']:
        path = folder / name
        source = path.read_text()
        source = source.replace('wandb.util.generate_id()', '__import__("uuid").uuid4().hex[:8]')
        source = source.replace('from torch.nn.attention.flex_attention import flex_attention, create_block_mask',
                                'from ax_attention_compat import flex_attention, create_block_mask')
        source = source.replace('from torch.nn.attention.flex_attention import BlockMask',
                                'from ax_attention_compat import BlockMask')
        if name == 'inference_mmu.py':
            marker = "        responses[j] += f'User: ' + question + f'\\n Answer : ' + text[0] + '\\n'"
            if '# AX_RESPONSE_JSON' not in source:
                if marker not in source:
                    raise RuntimeError('The upstream MMU output anchor changed.')
                source = source.replace(marker, marker + '\n        import json\n        print("AX_RESPONSE_JSON:" + json.dumps(text[0]), flush=True)  # AX_RESPONSE_JSON')
            source = source.replace('max_new_tokens=300,', 'max_new_tokens=config.get("max_new_tokens", 96),')
            if '    set_seed(config.get("seed", 42))' not in source:
                source = source.replace('    # VQ model for processing image into discrete tokens', '    set_seed(config.get("seed", 42))\n\n    # VQ model for processing image into discrete tokens')
        if name == 'inference_t2i.py':
            source = source.replace(')).to(torch.bfloat16).to(device)', ')).to(weight_type).to(device)')
            if '# AX_SEED' not in source:
                source = source.replace('    config = get_config()', '    config = get_config()\n    set_seed(config.get("seed", 42))  # AX_SEED')
            marker = '        wandb_images = [wandb.Image(image, caption=prompts[i]) for i, image in enumerate(checked_images)]'
            if '# AX_SAVE_IMAGES' not in source:
                if marker not in source:
                    raise RuntimeError('The upstream T2I output anchor changed.')
                source = source.replace(marker, '''        # AX_SAVE_IMAGES
        os.makedirs(config.experiment.output_dir, exist_ok=True)
        for image_index, generated_image in enumerate(checked_images):
            generated_image.save(os.path.join(config.experiment.output_dir, f"sample_{step + image_index:03d}.png"))
''' + marker)
        if name in ['inference_mmu.py', 'inference_t2i.py'] and '# AX_CPU_INT8' not in source:
            source = source.replace('    model.eval()', '''    model.eval()
    # AX_CPU_INT8: dynamic quantization of linear layers for the free CPU runtime.
    if device.type == "cpu" and config.get("cpu_int8", True):
        torch.backends.quantized.engine = "fbgemm"
        model = torch.ao.quantization.quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8, inplace=True)
        print("CPU linear weights: dynamic int8; other operations: float32", flush=True)''')
        path.write_text(source)
    path = folder / 'utils.py'
    source = path.read_text()
    if '# Image-only classroom runtime' not in source:
        source = source.replace('from decord import VideoReader, cpu',
            'try:\n    from decord import VideoReader, cpu\nexcept ImportError:\n    VideoReader = cpu = None  # Image-only classroom runtime')
    path.write_text(source)
    return folder



prepare(SHOWO_ROOT)
print("Pinned image inference scripts are ready.")

# Use half precision for the Colab GPU runtime.
for name in ["inference_mmu.py", "inference_t2i.py"]:
    script = SHOWO2_DIR / name
    text = script.read_text()
    if name == "inference_mmu.py":
        text = text.replace("weight_type = torch.float32", "weight_type = (torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16 if torch.cuda.is_available() else torch.float32)")
    if 'elif config.model.weight_type == "float16":' not in text:
        text = text.replace('    elif config.model.weight_type == "float32":', '    elif config.model.weight_type == "float16":\n        weight_type = torch.float16\n    elif config.model.weight_type == "float32":')
    text = text.replace("use_safetensors=False).to(device)", "use_safetensors=False, torch_dtype=weight_type).to(device)")
    script.write_text(text)


In [ ]:
briefing_question = (
    "Describe this pet image as a visual briefing for an image-generation model. "
    "Mention the animal, appearance, pose, background, and mood. "
    "Keep it concise."
)

print("Question for Show-o2 MMU:")
print(briefing_question)


In [ ]:
def run_showo2_mmu(image_path, question, config_name="configs/showo2_1.5b_demo_432x432.yaml"):
    cmd = [SHOWO_PYTHON, "inference_mmu.py", f"config={config_name}",
        f"mmu_image_path={Path(image_path).resolve()}", f"question={question}",
        f"model.vae_model.pretrained_model_path={SHOWO_VAE}", "max_new_tokens=160", "seed=42"]
    started = time.perf_counter()
    result = subprocess.run(cmd, cwd=SHOWO2_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode:
        print(result.stdout)
        result.check_returncode()
    answers = [line.split("AX_RESPONSE_JSON:", 1)[1] for line in result.stdout.splitlines() if line.startswith("AX_RESPONSE_JSON:")]
    if not answers:
        raise RuntimeError("Show-o2 returned no structured response.\n" + result.stdout[-3000:])
    answer = json.loads(answers[-1])
    if not answer.strip():
        raise RuntimeError("Show-o2 returned an empty answer.")
    print("Question:", question)
    print("Response:", answer)
    print(f"Measured MMU call: {time.perf_counter()-started:.1f}s")
    return answer

briefing_text = run_showo2_mmu(pet_image_path, briefing_question)


## Demo code

Follow the generated prompt into the image stage. Which input details survive?

In [ ]:
question = "Describe this pet and its visible background in one sentence."
briefing = run_showo2_mmu(pet_image_path, question)
print(briefing)
# The next experiment uses the actual model response as its image prompt.

In [ ]:
prompt_question = (
    "Based on this pet image, write a clean text-to-image prompt. "
    "Use this format only: Subject: ... Style: ... Scene: ... Details to preserve: ..."
)

print(prompt_question)


In [ ]:
image_grounded_prompt_text = run_showo2_mmu(pet_image_path, prompt_question)


In [ ]:
# Pass the actual model response to generation. No manual success substitute.
showo2_pet_prompt = image_grounded_prompt_text.strip()
assert showo2_pet_prompt
print("Actual image-grounded prompt:")
print(showo2_pet_prompt)


In [ ]:
SHOWO_CALL_NUMBER = 0

def run_showo2_t2i(prompts, config_name="configs/showo2_1.5b_demo_432x432.yaml", guidance_scale=5.0, num_inference_steps=25, batch_size=1):
    global SHOWO_CALL_NUMBER
    if isinstance(prompts, str):
        prompts = [prompts]
    if not prompts or any(not prompt.strip() for prompt in prompts):
        raise ValueError("Generation requires a non-empty prompt.")
    SHOWO_CALL_NUMBER += 1
    import uuid
    out_dir = (SHOWO2_OUTPUT_DIR / f"call_{SHOWO_CALL_NUMBER:02d}_{uuid.uuid4().hex[:8]}").resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    prompt_path = out_dir / "prompts.txt"
    prompt_path.write_text("\n".join(prompts))
    cmd = [SHOWO_PYTHON, "inference_t2i.py", f"config={config_name}",
        f"dataset.params.validation_prompts_file={prompt_path}",
        f"experiment.output_dir={out_dir}", f"guidance_scale={guidance_scale}",
        f"num_inference_steps={num_inference_steps}", f"batch_size={batch_size}",
        f"model.vae_model.pretrained_model_path={SHOWO_VAE}",
        f"model.weight_type={'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' if torch.cuda.is_available() else 'float32'}", "seed=42"]
    started = time.perf_counter()
    result = subprocess.run(cmd, cwd=SHOWO2_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode:
        print(result.stdout)
        result.check_returncode()
    generated = sorted(out_dir.glob("sample_*.png"))
    if len(generated) != len(prompts):
        raise RuntimeError(f"Expected {len(prompts)} images, found {len(generated)}.\n" + result.stdout[-3000:])
    print("Actual generation prompts:", prompts)
    print(f"Measured T2I call: {time.perf_counter()-started:.1f}s")
    return generated

showo2_generated_paths = run_showo2_t2i(showo2_pet_prompt, num_inference_steps=25)


In [ ]:
# Display generated Show-o2 image(s), if available.
if showo2_generated_paths:
    imgs = [Image.open(p).convert('RGB') for p in showo2_generated_paths[:4]]
    titles = [p.name for p in showo2_generated_paths[:4]]
    show_image_grid(imgs, titles=titles, cols=len(imgs), figsize=(12, 4), suptitle="Show-o2 generated image(s)")
else:
    print("No Show-o2 image files found yet. Run the T2I cell on a GPU runtime.")


In [ ]:
assistant_request = (
    "Use this pet as a children's book character. Reply with three short lines: "
    "TITLE: a title of up to six words. CONCEPT: one short sentence. "
    "IMAGE_PROMPT: one text-to-image prompt of at most fifty words preserving the pet's visible appearance."
)
assistant_text = run_showo2_mmu(pet_image_path, assistant_request)


In [ ]:
if "IMAGE_PROMPT:" not in assistant_text:
    raise ValueError("The model did not produce the required IMAGE_PROMPT field. Inspect the actual response above.")
assistant_visual_prompt = assistant_text.split("IMAGE_PROMPT:", 1)[1].strip()
assert assistant_visual_prompt
assistant_image_paths = run_showo2_t2i(assistant_visual_prompt, num_inference_steps=25)
img = Image.open(assistant_image_paths[0]).convert('RGB')
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis('off')
plt.title("Show-o2 assistant image from the actual text response")
plt.show()


In [ ]:
if SHOWO2_DIR.exists():
    possible_mixed_scripts = sorted(SHOWO2_DIR.glob("*mixed*.py")) + sorted(SHOWO2_DIR.glob("*interleave*.py"))
    if possible_mixed_scripts:
        print("Possible mixed-modality scripts found:")
        for p in possible_mixed_scripts:
            print("-", p)
    else:
        print("No obvious mixed-modality inference script found in this repo snapshot.")
        print("Use Application 4 as the stable Show-o2 workflow for this lab.")
else:
    print("Show-o2 repository is not cloned yet.")


In [ ]:
application_summary = {
    "Visual Briefing": {
        "Input": "Pet image + instruction",
        "Output": "Text briefing",
        "Show-o2 mode": "MMU / understanding",
    },
    "Prompt Writing from Image": {
        "Input": "Pet image + prompt-writing request",
        "Output": "Structured text prompt",
        "Show-o2 mode": "MMU / understanding",
    },
    "Visual Response Generation": {
        "Input": "Text prompt",
        "Output": "Generated image",
        "Show-o2 mode": "T2I / generation",
    },
    "Multimodal Assistant Output": {
        "Input": "Pet image + user request",
        "Output": "Text plan + generated image",
        "Show-o2 mode": "MMU + T2I",
    },
}

print(json.dumps(application_summary, indent=2, ensure_ascii=False))


## Interpretation

The chain must use the actual model response, not a manual substitute.

Compare the output with the evidence in the input. A generated answer can be wrong.